In [11]:
import pandas as pd
import numpy as np

np.random.seed(42)

dataset_path = '../assets/processed/'
X_train, X_dev, X_test, = pd.read_csv(f'{dataset_path}X_train.csv'), pd.read_csv(f'{dataset_path}X_dev.csv'), pd.read_csv(f'{dataset_path}X_test.csv')
Y_train, Y_dev, Y_test, = pd.read_csv(f'{dataset_path}Y_train.csv'), pd.read_csv(f'{dataset_path}Y_dev.csv'), pd.read_csv(f'{dataset_path}Y_test.csv')

X_train, X_dev, X_test = X_train.to_numpy(dtype='float32').T, X_dev.to_numpy(dtype='float32').T, X_test.to_numpy(dtype='float32').T # (n_features, n_samples)
Y_train, Y_dev, Y_test = Y_train.to_numpy(dtype='float32').T, Y_dev.to_numpy(dtype='float32').T, Y_test.to_numpy(dtype='float32').T # (1, n_samples)

### Parameters initialization

In [14]:
learning_rate = 0.001
batch_size = 128 # Minibatch size
epochs = 100
units = [X_train.shape[0], 128, 64, 32, 1] # input, hidden, output layers
activation = 'relu'
lambd = 0.1 # L2 regularization parameter
beta_momentum = 0.9 # momentum parameter
beta_rmsprop = 0.999 # RMSProp parameter
epsilon = 1e-8 # small value to avoid division by zero
weight_decay = 1e-2

parameters = {}
for layer in range(1, len(units)):
    # W: (layer units, layer-1 units)
    parameters[f'W{layer}'] = np.random.randn(units[layer], units[layer - 1]) * np.sqrt(2 / units[layer - 1]) # He initialization
    parameters[f'b{layer}'] = np.zeros((units[layer], 1))

batches = []
for i in range(0, X_train.shape[1], batch_size): # iterate over samples (axis 1)
    batches.append((X_train[:, i:i + batch_size], Y_train[:, i:i + batch_size]))

adam_parameters = {}
for layer in range(1, len(units)):
    adam_parameters[f'VdW{layer}'] = np.zeros_like(parameters[f'W{layer}'])
    adam_parameters[f'Vdb{layer}'] = np.zeros_like(parameters[f'b{layer}'])
    adam_parameters[f'SdW{layer}'] = np.zeros_like(parameters[f'W{layer}'])
    adam_parameters[f'Sdb{layer}'] = np.zeros_like(parameters[f'b{layer}'])

In [3]:
def activation_func(activaton, Z):
    if activaton == 'relu':
        return np.maximum(0, Z)
    if activaton == 'sigmoid':
        return 1 / (1 + np.exp(-Z))

In [4]:
def derivative_activation_func(activaton, Z):
    if activaton == 'relu':
        return np.where(Z > 0, 1, 0) # derivative of ReLU (1 if Z > 0 else 0)

In [5]:
def loss_func(Y_hat, Y, parameters, lambd):
    m = Y.shape[1]
    cross_entropy_loss = -1/m * np.sum(Y * np.log(Y_hat + 1e-8) + (1 - Y) * np.log(1 - Y_hat + 1e-8)) # epsilon to avoid log(0)
    l2_regularization_loss = (lambd / (2 * m)) * sum(np.sum(np.square(parameters[f'W{layer}'])) for layer in range(1, len(units))) # frobenius norm of weights
    return cross_entropy_loss + l2_regularization_loss

In [6]:
def forward_propagation(X, L, activation, parameters):
    cache = {}
    A = X # A0 (input layer)

    for layer in range(1, L + 1):
        W = parameters[f'W{layer}']
        b = parameters[f'b{layer}']
        
        Z = np.dot(W, A) + b # Z = W * A + b
        A = activation_func(activation, Z) if layer < L else activation_func('sigmoid', Z) # sigmoid for the output layer
        cache[f'Z{layer}'], cache[f'A{layer}'] = Z, A

    return A, cache

In [7]:
def backward_propagation(X, dZ, m, L, activation, parameters, cache):
    grads = {}

    for layer in reversed(range(1, L + 1)):
        A_prev = cache[f'A{layer - 1}'] if layer > 1 else X
        W = parameters[f'W{layer}']
        
        grads[f'dW{layer}'] = (1 / m) * np.dot(dZ, A_prev.T)
        grads[f'db{layer}'] = (1 / m) * np.sum(dZ, axis=1, keepdims=True)

        if layer > 1:
            dA_prev = np.dot(W.T, dZ)
            dZ = dA_prev * derivative_activation_func(activation, cache[f'Z{layer - 1}'])

    return grads

In [8]:
def evaluate(X, Y, L, activation, parameters, lambd):
    Y_hat, _ = forward_propagation(X, L, activation, parameters)
    loss = loss_func(Y_hat, Y, parameters, lambd)
    accuracy = ((Y_hat > 0.5) == Y).mean() 
    return loss, accuracy

### Model training

In [ ]:
t = 0 # ADAM optimization time step
for epoch in range(epochs):
    for X_batch, Y_batch in batches:
        m = X_batch.shape[1] # samples in this batch
        L = len(units) - 1

        # forward prop
        Y_hat, cache = forward_propagation(X_batch, L, activation, parameters)

        # loss with regularization
        loss = loss_func(Y_hat, Y_batch, parameters, lambd)

        # back prop
        dZ = Y_hat - Y_batch # sigmoid + BCE chain derivative (last layer)
        grads = backward_propagation(X_batch, dZ, m, L, activation, parameters, cache)

        # Update parameters with L2 regularization
        t += 1
        for layer in range(1, L + 1):
            dW = grads[f'dW{layer}']
            db = grads[f'db{layer}']
            
            # momentum and RMSProp updates
            adam_parameters[f'VdW{layer}'] = beta_momentum * adam_parameters[f'VdW{layer}'] + (1 - beta_momentum) * dW
            adam_parameters[f'Vdb{layer}'] = beta_momentum * adam_parameters[f'Vdb{layer}'] + (1 - beta_momentum) * db
            adam_parameters[f'SdW{layer}'] = beta_rmsprop * adam_parameters[f'SdW{layer}'] + (1 - beta_rmsprop) * (dW ** 2)
            adam_parameters[f'Sdb{layer}'] = beta_rmsprop * adam_parameters[f'Sdb{layer}'] + (1 - beta_rmsprop) * (db ** 2)

            # bias correction
            VdW_corr = adam_parameters[f'VdW{layer}'] / (1 - beta_momentum ** t)
            Vdb_corr = adam_parameters[f'Vdb{layer}'] / (1 - beta_momentum ** t)
            SdW_corr = adam_parameters[f'SdW{layer}'] / (1 - beta_rmsprop ** t)
            Sdb_corr = adam_parameters[f'Sdb{layer}'] / (1 - beta_rmsprop ** t)

            dW_adam = VdW_corr / (np.sqrt(SdW_corr) + epsilon) # ADAM update for weights
            db_adam = Vdb_corr / (np.sqrt(Sdb_corr) + epsilon) # ADAM update for bias

            parameters[f'W{layer}'] -= learning_rate * (dW_adam + weight_decay * parameters[f'W{layer}'])
            parameters[f'b{layer}'] -= learning_rate * db_adam

    if epoch % 20 == 0:
        tr_loss, tr_acc = evaluate(X_train, Y_train, L, activation, parameters, lambd)
        dv_loss, dv_acc = evaluate(X_dev,   Y_dev,   L, activation, parameters, lambd)
        print(f'ep {epoch:3d}  '
                f'train: | loss {tr_loss:.4f} acc {tr_acc:.4f} | '
                f'dev: | loss {dv_loss:.4f} acc {dv_acc:.4f} |')

ep   0  train: | loss 0.6503 acc 0.6199 | dev: | loss 0.6611 acc 0.6104 |
ep  20  train: | loss 0.6344 acc 0.6367 | dev: | loss 0.6677 acc 0.6028 |
ep  40  train: | loss 0.6255 acc 0.6443 | dev: | loss 0.6855 acc 0.5939 |
ep  60  train: | loss 0.6193 acc 0.6510 | dev: | loss 0.7016 acc 0.5864 |
ep  80  train: | loss 0.6163 acc 0.6539 | dev: | loss 0.7084 acc 0.5824 |
